# SQ-5 — Scalar (ε, AUROC) reporting vs the stepwise privacy card

**Tests:** is the privacy claim that scalar (ε, AUROC) reporting *says* the same as the privacy claim the stepwise card *shows*? Concretely: does the scalar ordering across FL configurations match the empirical attack-rate ordering, and does the stepwise card carry information the scalar form does not?

**Headline.** A side-by-side table for three FL configurations:
- **Loose**: σ=0.3, no clipping discipline → large effective ε
- **Default**: σ=1.1, clip_norm=1.0 → moderate ε (the example card's configuration)
- **Tight**: σ=2.5, clip_norm=1.0 → small ε

For each: report the scalar (ε from the RDP accountant, AUROC on the released model), then the stepwise card's worst-record TPR at FPR=10⁻³.

**Claim being tested:** the scalar form ranks configurations by ε; the calibrated form surfaces *per-record* attack rates that ε alone cannot rank. The two forms can — and do — *disagree on operational risk ordering* once clipping and noise interact.

> Tutorial scale runs in 1–3 min on a workstation (three small FL training runs + three small MIA sweeps). Paper-scale increases shadow counts.

In [1]:
import json, sys, time
from pathlib import Path

import numpy as np

EVAL_DIR = Path.cwd().parents[1] if Path.cwd().name.startswith('sq') else Path.cwd()
sys.path.insert(0, str(EVAL_DIR))

from flta_eval import audit, attacks, datasets, fl

In [2]:
PAPER_SCALE = False
MASTER_SEED = 20260525

configs = [
    ('Loose',   fl.FLConfig(n_rounds=15, client_lr=0.1, client_batch_size=32,
                            clip_norm=10.0, noise_multiplier=0.3)),
    ('Default', fl.FLConfig(n_rounds=15, client_lr=0.1, client_batch_size=32,
                            clip_norm=1.0,  noise_multiplier=1.1)),
    ('Tight',   fl.FLConfig(n_rounds=15, client_lr=0.1, client_batch_size=32,
                            clip_norm=1.0,  noise_multiplier=2.5)),
]

n_targets, n_shadow = (40, 64) if PAPER_SCALE else (6, 12)

ds = datasets.load_bloodmnist(EVAL_DIR / 'data')
manifest = datasets.load_partition(EVAL_DIR / 'data')
pod_data = [
    (ds['X_train'][np.array(manifest['pod_indices'][i])],
     ds['y_train'][np.array(manifest['pod_indices'][i])])
    for i in range(manifest['n_pods'])
]
print(f'n_targets={n_targets}, n_shadow={n_shadow}, fl pods={len(pod_data)}')

n_targets=6, n_shadow=12, fl pods=50


In [3]:
rows = []
for label, cfg in configs:
    t0 = time.time()
    model = fl.MLP(input_dim=ds['input_dim'], hidden_dim=64, n_classes=ds['n_classes'])
    model.init_from_seed(MASTER_SEED, f'fl.scalarvs.{label}')
    trained, _ = fl.federated_train(
        model=model, pod_data=pod_data,
        X_test=ds['X_test'], y_test=ds['y_test'], config=cfg,
        master_seed=MASTER_SEED, namespace=f'fl.scalarvs.{label}.train',
    )
    train_t = time.time() - t0

    acc = trained.accuracy(ds['X_test'], ds['y_test'])
    eps = fl.rdp_epsilon(noise_multiplier=cfg.noise_multiplier,
                         n_rounds=cfg.n_rounds, sample_rate=1.0, delta=1e-5)

    t0 = time.time()
    mia = attacks.per_record_mia(
        federated_model=trained, pod_data=pod_data,
        X_test=ds['X_test'], y_test=ds['y_test'],
        n_targets=n_targets, n_shadow_runs=n_shadow,
        shadow_steps=15, shadow_batch_size=32,
        fpr_targets=(0.001, 0.01),
        master_seed=MASTER_SEED, namespace=f'attacks.mia.scalarvs.{label}',
    )
    mia_t = time.time() - t0

    rows.append({
        'label': label,
        'sigma': cfg.noise_multiplier,
        'clip_norm': cfg.clip_norm,
        'rounds': cfg.n_rounds,
        'rdp_epsilon': float(eps),
        'test_accuracy': float(acc),
        'worst_tpr': float(mia.worst_record_tpr_at_fpr),
        'median_tpr': float(mia.median_tpr_at_fpr),
        'train_s': round(train_t, 1),
        'mia_s': round(mia_t, 1),
    })

print(f"{'config':>8s} {'σ':>5s} {'clip':>6s} {'ε':>8s} {'AUROC':>7s} {'worst TPR':>10s} {'median TPR':>11s}")
for r in rows:
    print(f"{r['label']:>8s} {r['sigma']:>5.2f} {r['clip_norm']:>6.1f}"
          f" {r['rdp_epsilon']:>8.2f} {r['test_accuracy']:>7.4f}"
          f" {r['worst_tpr']:>10.4f} {r['median_tpr']:>11.4f}")

  config     σ   clip        ε   AUROC  worst TPR  median TPR
   Loose  0.30   10.0   273.03  0.1386     0.2500      0.1458
 Default  1.10    1.0    36.31  0.1856     0.6250      0.2833
   Tight  2.50    1.0    12.96  0.1520     0.2000      0.0833


In [4]:
# Compare the two orderings
by_epsilon = sorted(rows, key=lambda r: r['rdp_epsilon'])
by_worst_tpr = sorted(rows, key=lambda r: r['worst_tpr'])

print('Scalar ordering (low ε = more private):')
for r in by_epsilon:
    print(f"  {r['label']:8s}  ε={r['rdp_epsilon']:7.2f}")
print()
print('Empirical ordering (low worst-record TPR = less leakage):')
for r in by_worst_tpr:
    print(f"  {r['label']:8s}  worst TPR={r['worst_tpr']:.4f}")
print()
scalar_seq = [r['label'] for r in by_epsilon]
empirical_seq = [r['label'] for r in by_worst_tpr]
agree = scalar_seq == empirical_seq
print(f'Orderings agree: {agree}')
if not agree:
    print('  → scalar reporting and empirical calibration disagree on the operational ordering.')
    print('  → the card surfaces what ε alone hides: per-record attack rates are not monotonic in ε under realistic FL configs.')
else:
    print('  → orderings agree at this configuration; the calibration block still adds the *per-record* tail information that the scalar form does not carry.')

Scalar ordering (low ε = more private):
  Tight     ε=  12.96
  Default   ε=  36.31
  Loose     ε= 273.03

Empirical ordering (low worst-record TPR = less leakage):
  Tight     worst TPR=0.2000
  Loose     worst TPR=0.2500
  Default   worst TPR=0.6250

Orderings agree: False
  → scalar reporting and empirical calibration disagree on the operational ordering.
  → the card surfaces what ε alone hides: per-record attack rates are not monotonic in ε under realistic FL configs.


In [5]:
record_path = audit.write_result_record(
    target_dir=EVAL_DIR / 'results' / 'sq5',
    attack='scalar_vs_stepwise',
    variant='SQ-5',
    threat_profile='R',
    dataset={'name': 'bloodmnist', 'sha256': manifest['npz_sha256']},
    config={
        'paper_scale': PAPER_SCALE,
        'n_targets_per_config': n_targets, 'n_shadow_runs_per_config': n_shadow,
        'configs': [{'label': lbl, 'sigma': c.noise_multiplier, 'clip_norm': c.clip_norm,
                     'rounds': c.n_rounds} for lbl, c in configs],
    },
    seed_namespace='sq5.scalar_vs_stepwise.bloodmnist.v1',
    result={
        'rows': rows,
        'scalar_ordering': scalar_seq,
        'empirical_ordering': empirical_seq,
        'orderings_agree': agree,
    },
    tolerances={'note': 'qualitative — what each reporting form surfaces matters as much as exact value agreement'},
)
print(f'wrote {record_path.relative_to(EVAL_DIR)}')

wrote results/sq5/scalar_vs_stepwise__SQ-5__2026-05-26T08-56-41Z.json


## Reading the result

Two things to take from the table.

**1. Per-record information is not in ε.** Even when the scalar ordering and the empirical ordering agree, scalar (ε, AUROC) reports a population-average summary. The stepwise card carries the worst-record TPR, the median, and the FPR threshold at which they were measured. A reviewer looking at the card knows that the *worst* record in this configuration faces a particular attack rate; a reviewer looking at the scalar pair does not.

**2. The orderings can disagree.** When clipping is permissive (the *Loose* config) the RDP-accountant's reported ε does not bound what gradient inversion or MIA actually achieve against the released model. The card surfaces the gap; the scalar pair conceals it.

**Why this is the SQ that the paper hangs on.** SQ-1 measures the calibration; SQ-2 measures the metadata; SQ-3 measures the rule engine; SQ-5 measures *what each reporting form lets a reviewer see*. This is the head-to-head the paper's contribution claim depends on.